# Preprocessing Others Utilities Demo

Demostración interactiva de las funciones utilitarias auxiliares de validación de esquema, manipulación segura de índices, detección y conversión polimórfica de marcas de tiempo y resumen diagnóstico de columnas implementadas en `_preprocessing_others.py` y expuestas por `ds_utils.preprocessing.preprocessing_utilities`.

Funciones cubiertas en este notebook:
- `validate_required_columns`
- `set_index_safe`
- `reset_index_safe`
- `detect_timestamp`
- `convert_timestamp`
- `get_columns_summary`

## Imports

In [1]:
# RECARGA AUTOMÁTICA DE MÓDULOS EXTERNOS
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from ds_utils.preprocessing.preprocessing_utilities import (
    validate_required_columns,
    set_index_safe,
    reset_index_safe,
    detect_timestamp,
    convert_timestamp,
    get_columns_summary,
)

## validate_required_columns

`validate_required_columns` comprueba de forma segura la presencia de todas las columnas obligatorias requeridas en un DataFrame, lanzando un `KeyError` con los nombres faltantes si alguna está ausente.

In [2]:
df_schema = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "country": ["ES", "FR", "DE"]
})

# 1. Validación satisfactoria
validate_required_columns(df=df_schema, required_cols=["customer_id", "country"])
print("Validación de columnas existentes: OK")

# 2. Captura de columnas faltantes
try:
    validate_required_columns(df=df_schema, required_cols=["customer_id", "email"])
except KeyError as exc:
    print("Error esperado por columnas ausentes capturado:", exc)

Validación de columnas existentes: OK
Error esperado por columnas ausentes capturado: "Missing columns in the DataFrame: ['email']. Check for typos, spaces, or capitalization."


## set_index_safe

`set_index_safe` establece una columna como índice garantizando idempotencia: si la columna ya forma parte del índice, devuelve el DataFrame sin relanzar excepciones.

In [3]:
df_idx_test = pd.DataFrame({
    "record_id": ["R1", "R2", "R3"],
    "score": [10, 20, 30]
})

df_indexed = set_index_safe(df=df_idx_test, column="record_id")
print("DataFrame tras primera asignación de índice:")
display(df_indexed)

# Segunda llamada idempotente
df_reindexed = set_index_safe(df=df_indexed, column="record_id")
print("¿Se mantiene idéntico sin error?:", df_reindexed.equals(df_indexed))

DataFrame tras primera asignación de índice:


,score
record_id,
R1,10
R2,20
R3,30


¿Se mantiene idéntico sin error?: True


## reset_index_safe

`reset_index_safe` restaura índices personalizados como columnas ordinarias y, si el DataFrame ya cuenta con un `RangeIndex`, evita generar columnas no deseadas como `'index'` o `'level_0'`.

In [4]:
# 1. Restaurar índice personalizado
df_restored = reset_index_safe(df=df_indexed)
print("Índice personalizado restaurado a columna:")
display(df_restored)

# 2. Reset seguro sobre RangeIndex
df_range_safe = reset_index_safe(df=df_restored)
print("Columnas tras reset sobre RangeIndex (sin columna 'index' espuria):", list(df_range_safe.columns))

Índice personalizado restaurado a columna:


,record_id,score
0,R1,10
1,R2,20
2,R3,30


Columnas tras reset sobre RangeIndex (sin columna 'index' espuria): ['record_id', 'score']


## detect_timestamp

`detect_timestamp` examina e interpreta representaciones temporales (epoch en segundos o milisegundos, ISO 8601 o formato desconocido) y devuelve una descripción legible con la fecha UTC correspondiente.

In [5]:
timestamp_examples = [
    1704067200,                  # Epoch segundos
    1704067200000,               # Epoch milisegundos
    "2024-01-01T15:30:00Z",      # ISO 8601
    "fecha_invalida"
]

for ts in timestamp_examples:
    print(f"{ts} -> {detect_timestamp(ts)}")

1704067200 -> Epoch in seconds: 2024-01-01 00:00:00 UTC
1704067200000 -> Epoch in milliseconds: 2024-01-01 00:00:00 UTC
2024-01-01T15:30:00Z -> ISO 8601 format: 2024-01-01 15:30:00+00:00 (timezone included if present)
fecha_invalida -> Unknown timestamp format


## convert_timestamp

`convert_timestamp` convierte de forma polimórfica marcas temporales en segundos, milisegundos, microsegundos, nanosegundos o cadenas ISO 8601 a `pd.Timestamp` (o `pd.NaT` si no es válido).

In [6]:
raw_timestamps = [1704067200, 1704067200000, 1704067200000000000, "2024-01-01 00:00:00", "error"]
converted_timestamps = [convert_timestamp(t) for t in raw_timestamps]

df_conversions = pd.DataFrame({
    "entrada": [str(t) for t in raw_timestamps],
    "convertido": converted_timestamps,
    "tipo": [type(c).__name__ for c in converted_timestamps]
})
display(df_conversions)

,entrada,convertido,tipo
0,1704067200,2024-01-01 00:00:00,Timestamp
1,1704067200000,2024-01-01 00:00:00,Timestamp
2,1704067200000000000,2024-01-01 00:00:00,Timestamp
3,2024-01-01 00:00:00,2024-01-01 00:00:00+00:00,Timestamp
4,error,NaT,NaTType


## get_columns_summary

`get_columns_summary` genera una tabla diagnóstica con nombre, tipo de dato, cantidad de valores distintos, número de celdas no válidas y porcentaje de datos ausentes para cada columna.

In [8]:
df_summary_sample = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "category": ["A", "B", "A", "B", "C"],
    "temperature": [21.5, np.nan, 23.0, np.nan, 22.1],    # 2 nulos (40%)
    "comments": ["Bien", "   ", "", "Excelente", np.nan],  # 3 vacíos (60%)
    "active": [True, True, False, True, False]
})

df_col_summary = get_columns_summary(
    df=df_summary_sample,
    sort_by="porcentaje_no_validos",
    ascending=False
)

print("Resumen diagnóstico ordenado por porcentaje de nulos:")
display(df_col_summary)

Resumen diagnóstico ordenado por porcentaje de nulos:


,nombre_columna,tipo_dato,num_valores_distintos,num_valores_no_validos,porcentaje_no_validos
0,comments,str,4,3,60.0
1,temperature,float64,3,2,40.0
2,id,int64,5,0,0.0
3,category,str,3,0,0.0
4,active,bool,2,0,0.0
